# Chương 3 — Ứng dụng 1: Market Basket Analysis

**Thuật toán:** negFIN (Julia, cài đặt của nhóm). **Dữ liệu thực:** Groceries — 9.835 giao dịch mua sắm, 169 nhóm sản phẩm. Không sử dụng thư viện FIM có sẵn.

Quy trình: tải CSV → mỗi dòng là một giao dịch → chuẩn hóa nhãn sản phẩm → ánh xạ sang ID nguyên → chạy `src/application_mining.jl` → sinh luật X→Y từ itemset của negFIN → xếp hạng top-10 theo lift → phân tích ý nghĩa kinh doanh.

Nguồn dữ liệu: [arules – Groceries](https://michael.hahsler.net/arules/reference/Groceries.html). File CSV được tải khi chạy notebook từ repo công khai đã ghi ở `applications/market_basket.py`.

**Lưu ý Colab:** chỉnh `REPO_URL` trong ô tiếp theo thành URL GitHub repository nhóm sau khi push; nếu chạy Jupyter ngay từ thư mục repo, để rỗng.

In [ ]:
# Tự dùng code trong repository hoặc clone repo chính thức (sau khi đã push Chương 3).
from pathlib import Path
import os, subprocess, sys, shutil, platform

REPO_URL = "https://github.com/Thanhtung46/Frequent_Itemset_Mining.git"
PROJECT_DIR = None
for directory in [Path.cwd(), Path.cwd() / 'Frequent_Itemset_Mining', Path('/content/Frequent_Itemset_Mining')]:
    if (directory / 'src' / 'algorithm' / 'negfin_baseline.jl').exists() and (directory / 'applications').is_dir():
        PROJECT_DIR = directory.resolve()
        break
if PROJECT_DIR is None:
    if not REPO_URL.startswith('https://github.com/'):
        raise ValueError('Đặt REPO_URL tới repo GitHub public của nhóm.')
    PROJECT_DIR = Path('/content/Frequent_Itemset_Mining')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
if not (PROJECT_DIR / 'applications' / 'common.py').exists():
    raise RuntimeError('Repo GitHub chưa được push phần Chương 3. Hãy commit/push mã nguồn mới.')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Project directory:', PROJECT_DIR)


In [ ]:
# Cài gói Python hỗ trợ đọc dữ liệu/biểu đồ/ảnh; KHÔNG cài gói khai thác FIM.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-ch3.txt'], check=True)

# Cài Julia nếu đang dùng Colab Linux. Máy Windows/Mac: cài Julia >= 1.9 trước.
if shutil.which(os.environ.get('JULIA_BIN', 'julia')) is None:
    if sys.platform == 'linux' and platform.machine() in ('x86_64', 'AMD64'):
        julia_home = Path('/content/julia-1.11.4') if Path('/content').is_dir() else Path.home() / '.cache' / 'negfin-julia-1.11.4'
        julia_exe = julia_home / 'bin' / 'julia'
        if not julia_exe.exists():
            command = ('curl -fsSL https://julialang-s3.julialang.org/bin/linux/x64/1.11/'
                       'julia-1.11.4-linux-x86_64.tar.gz -o /tmp/negfin-julia.tar.gz'
                       ' && mkdir -p ' + str(julia_home)
                       + ' && tar -xzf /tmp/negfin-julia.tar.gz -C ' + str(julia_home)
                       + ' --strip-components=1')
            subprocess.run(['bash', '-lc', command], check=True)
        os.environ['JULIA_BIN'] = str(julia_exe)
    else:
        raise RuntimeError('Cần cài Julia >= 1.9 và đưa julia vào PATH hoặc đặt biến JULIA_BIN.')
subprocess.run([os.environ.get('JULIA_BIN', 'julia'), '--version'], check=True)
print('Dùng negFIN của nhóm:', PROJECT_DIR / 'src' / 'algorithm' / 'negfin_baseline.jl')

## 1. Chuẩn bị dữ liệu và giải thích biểu diễn giao dịch

Mỗi hóa đơn tương ứng một giao dịch. Một mặt hàng chỉ xuất hiện một lần trong hóa đơn ngay cả khi được mua nhiều đơn vị (binary presence); loại bỏ ô CSV rỗng. Tên sản phẩm ánh xạ thành ID số nguyên ổn định theo thứ tự từ điển. **Không sử dụng thông tin bên ngoài giao dịch để khai thác frequent itemsets.**

In [ ]:
from applications.common import ROOT, save_transactions
from applications.market_basket import get_groceries_csv, load_groceries, encode_baskets
from IPython.display import display
raw_csv = get_groceries_csv(ROOT / 'data' / 'application' / 'groceries.csv')
baskets = load_groceries(raw_csv)
numeric, id_to_name = encode_baskets(baskets)
print('Số giao dịch:', len(numeric), '| Số loại hàng:', len(id_to_name))
for idx in range(3):
    print('Giỏ', idx+1, baskets[idx], '=>', numeric[idx])

## 2. Khai thác itemset, sinh luật và kiểm chứng support

- `minsup = 0.01`: tối thiểu 1% số hóa đơn chứa itemset.
- `minconf = 0.15`: xác suất mua vế phải khi có vế trái từ 15% trở lên.
- `lift = confidence(X→Y)/support(Y)`: mức xuất hiện cùng nhau so với kỳ vọng độc lập. Lift lớn hơn 1 thể hiện liên hệ dương, **không có nghĩa là quan hệ nhân quả**.

**Đảm bảo đúng hướng dẫn:** các frequent itemset dùng để sinh luật đều đọc từ file được `negfin_baseline` (Julia) tạo ra. Không gọi Apriori/FP-Growth ở Python.

In [ ]:
from applications.run_market import execute
MIN_SUPPORT = 0.01
MIN_CONFIDENCE = 0.15
market_result = execute(minsup=MIN_SUPPORT, minconf=MIN_CONFIDENCE, raw_csv=raw_csv)
display(market_result['rules'].head(10))

## 3. Top-10 theo lift, hình minh họa và diễn giải

Khi xếp theo lift, vẫn cần kiểm tra support và confidence để tránh hiểu sai luật ít phổ biến. Luật X→Y không nhất thiết đối xứng với Y→X do confidence khác nhau. Các gợi ý như bán kèm, bố trí kệ, marketing chỉ là giả thuyết kinh doanh cần A/B test; không thể suy luận lợi nhuận từ giao dịch thiếu giá bán/lãi gộp.

In [ ]:
from IPython.display import Image, Markdown
folder = market_result['work_dir']
display(Image(filename=str(folder / 'top_products.png')))
if (folder / 'top10_lift.png').exists():
    display(Image(filename=str(folder / 'top10_lift.png')))
from applications.market_basket import describe_business_rule
for i, (_, rule) in enumerate(market_result['rules'].head(10).iterrows(), start=1):
    display(Markdown(f'**Luật {i}:** {describe_business_rule(rule)}'))

## 4. Kết quả cần đưa vào báo cáo

Đính kèm bảng top-10 (`results/chapter3/market_basket/top10_by_lift.csv`), ngưỡng minsup/minconf, số luật, số itemset, hình top10 lift và nhận xét các ví dụ kèm **support, confidence, lift**. Trình bày nguồn dữ liệu thật, cách chuẩn hóa hóa đơn và giới hạn không có dữ liệu doanh thu, giá bán, thời gian hay thử nghiệm nhân quả.